In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,FloatType
from pyspark.sql import functions as F

catalog_name='ecommerce'

df_bronze=spark.table(f"{catalog_name}.bronze.brz_brands")
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name'))) #Removing spaces in the brand_name column

df_silver=df_silver.withColumn('brand_code',F.regexp_replace(F.col('brand_code'),r'[^a-zA-Z0-9]','')) # Replace all non alphanumeric characters with empty string
#df_sliver.show()

In [0]:
#df_sliver.select('category_code').distinct().show()


In [0]:
 
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])

# Show results
df_silver.show()

In [0]:
#write raw data into silver layes (catalog :ecommerce schema:silver table : slv_brands)
df_silver.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")


In [0]:
#category Table 

df_silver_cat=spark.table(f"{catalog_name}.bronze.brz_category")

# Get row and column count
row_count, column_count = df_silver_cat.count(), len(df_silver_cat.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_silver_cat=df_silver_cat.withColumn('category_code',F.upper(F.col('category_code'))) # convert category code to Upper Case by using Upper()

# df_silver_cat=df_silver_cat.groupBy(F.col('category_code')).count().filter(F.col('count')>1)
df_silver_cat=df_silver_cat.dropDuplicates(['category_code'])
df_silver_cat.show()

#write raw data into silver layes (catalog :ecommerce schema:silver table : slv_category)
df_silver_cat.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f"{catalog_name}.silver.slv_category")


In [0]:
df_silver_product=spark.table(f"{catalog_name}.bronze.brz_products")


row_count,column_count = df_silver_product.count(),len(df_silver_product.columns)

print(f"Row count :{row_count}")
print(f"Column count :{column_count}")

# upper the category code  ,brand code
df_silver_product=df_silver_product.withColumn("category_code",
                                                F.upper(F.col("category_code"))) \
                                    .withColumn("brand_code", 
                                                F.upper(F.col("brand_code")))
# Replace ,cast weight_grams and length-cm columns to integer type ,float Type and g, , with respectively 
df_silver_product=df_silver_product.withColumn('weight_grams',
                                               F.regexp_replace(F.col('weight_grams'),'g','')
                                               .cast(IntegerType()))\
                                    .withColumn('length_cm',
                                                F.regexp_replace(F.col('length_cm'),',','.')
                                                .cast(FloatType()))
df_silver_product = df_silver_product.withColumn(
    'material',
    F.when(F.col('material') == 'Coton', 'Cotton')
     .when(F.col('material') == 'Ruber', 'Rubber')
     .when(F.col('material') == 'Alumium', 'Aluminium')
     .otherwise(F.col('material'))
)
df_silver_product=df_silver_product.withColumn('rating_count',
                                               F.when(F.col('rating_count').isNotNull(),F.abs(F.col('rating_count')))
                                               .otherwise(F.lit(0))
                                               )
df_silver_product.show()
#write raw data into silver layes (catalog :ecommerce schema:silver table : slv_products)
df_silver_product.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f"{catalog_name}.silver.slv_products")





In [0]:

df_silver_cus=spark.table(f"{catalog_name}.bronze.brz_customers")

row_count,coulmn_count= df_silver_cus.count(),len(df_silver_cus.columns)

print(f"Row count :{row_count}")
print(f"Coulmn count:{coulmn_count}")

#Drop the Null Value rows of customer_id column

df_silver_cus=df_silver_cus.dropna(subset='customer_id')
row_count = df_silver_cus.count()
print(f"Row count after droping null values: {row_count}")


# Replace Null values in Phone column with Not Avaiable
df_silver_cus=df_silver_cus.fillna({'phone':'Not Available'})
df_silver_cus.show()
df_silver_cus.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f"{catalog_name}.silver.slv_customers")


In [0]:
catalog_name='ecommerce'
from pyspark.sql import functions as F

df_silver_date = spark.read.table(f"{catalog_name}.bronze.brz_date")

row_count, column_count = df_silver_date.count(), len(df_silver_date.columns)
columns = df_silver_date.columns

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")
print(f"Columns: {columns}")

df_silver_date = df_silver_date.withColumn('date', F.to_date(F.col('date'), 'dd-MM-yyyy'))\
    .withColumn('day_name',F.initcap(F.col('day_name')))
duplicates = df_silver_date.groupBy('date').count().filter("count > 1")
df_silver_date=df_silver_date.dropDuplicates(['date'])
df_silver_date=df_silver_date.withColumn('week_of_year',F.abs(F.col('week_of_year')))
df_silver_date=df_silver_date.withColumn('quater',F.concat_ws("",F.concat(F.lit('Q'),F.col('quater'),F.lit('-'),F.col('year'))))
df_silver_date=df_silver_date.withColumn('week_of_year',F.concat_ws("-",F.concat(F.lit('Week'),F.col('week_of_year'),F.lit('-'),F.col('year'))))
df_silver_date=df_silver_date.withColumnRenamed('week_of_year','week')
df_silver_date.show()
df_silver_date.write.format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema',True) \
    .saveAsTable(f"{catalog_name}.silver.slv_date")